# 음식 사진 배경 교체: 코랩 전체 검증

음식 사진 업로드부터 전경 분리, 빈 광고 배경 생성, 합성, 결과 확인까지 실행합니다. 코랩의 `venv` 기능에 의존하지 않고 프로젝트 전용 패키지 폴더를 사용하므로, `ensurepip` 누락으로 가상환경 생성이 실패하는 런타임에서도 실행할 수 있습니다.

In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# 반드시 final_1_team 내부의 새 배경 교체 프로젝트를 지정합니다.
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'프로젝트 폴더를 찾을 수 없습니다: {PROJECT_ROOT}'
%cd $PROJECT_ROOT
print(f'프로젝트 위치: {PROJECT_ROOT}')

Mounted at /content/drive
/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline
프로젝트 위치: /content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline


In [ ]:
import os
import subprocess
import sys

# 전역 사이트 패키지를 변경하지 않습니다. 의존성은 코랩 로컬의 전용 폴더에만 설치합니다.
PACKAGE_DIR = Path('/content/food-image-cleanup-packages')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--target', str(PACKAGE_DIR), '--prefer-binary', '--upgrade-strategy', 'only-if-needed', '--timeout', '60', '--retries', '2', '-r', 'requirements-colab.txt'], check=True)

# 이후 모델 다운로드·추론 프로세스에만 전용 패키지를 우선 적용합니다.
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['PYTHONPATH'] = str(PACKAGE_DIR) + os.pathsep + RUNTIME_ENV.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-c', "import open_clip, torch; print('OpenCLIP 설치 확인:', open_clip.__name__); print('CUDA 사용 가능:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')"], check=True, env=RUNTIME_ENV)

CompletedProcess(args=['/usr/bin/python3', '-c', "import open_clip, torch; print('OpenCLIP 설치 확인:', open_clip.__name__); print('CUDA 사용 가능:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')"], returncode=0)

In [ ]:
def run_or_show(command, *, env):
    result = subprocess.run(command, text=True, capture_output=True, env=env)
    if result.stdout:
        print(result.stdout)
    if result.returncode:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'명령 실행 실패({result.returncode}): {command}')
    return result


In [ ]:
# 선택값: sana-1.6b(기본, 토큰 불필요) 또는 flux-schnell(토큰 필요)
BACKGROUND_PROVIDER = 'sana-1.6b'
assert BACKGROUND_PROVIDER in {'sana-1.6b', 'flux-schnell'}

# 실행 설정도 선택한 배경 생성기로 맞춥니다.
import yaml
config_path = Path('configs/pipeline.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
config['models']['background_generator']['provider'] = BACKGROUND_PROVIDER
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('선택한 배경 생성기:', BACKGROUND_PROVIDER)

public_models = ['yolo', 'sam2', 'big-lama', 'openclip', 'birefnet']
if BACKGROUND_PROVIDER == 'sana-1.6b':
    public_models.append('sana')
run_or_show([sys.executable, '-m', 'scripts.download_models', '--models', *public_models], env=RUNTIME_ENV)
public_required = ['models/yolo11n.pt', 'models/sam2.1_t.pt', 'models/big-lama.pt', 'models/birefnet/config.json']
if BACKGROUND_PROVIDER == 'sana-1.6b':
    public_required.append('models/sana-1.6b/model_index.json')
missing = [path for path in public_required if not Path(path).is_file()]
assert not missing, f'공개 모델 누락: {missing}'
print('공개 모델 확인 완료')

선택한 배경 생성기: sana-1.6b
[SKIP] models/yolo11n.pt
[SKIP] models/sam2.1_t.pt
[SKIP] models/big-lama.pt
[DOWNLOAD] OpenCLIP ViT-B-32 (laion2b_s34b_b79k)
[SKIP] models/birefnet
[SKIP] models/sana-1.6b
[DONE] Model download completed

공개 모델 확인 완료


## FLUX.1 Schnell 접근 권한

위 선택값을 `flux-schnell`로 바꾼 경우에만 이 셀에서 Hugging Face 토큰을 입력합니다. Sana 1.6B를 선택한 경우 이 셀은 자동으로 건너뜁니다.

In [ ]:
if BACKGROUND_PROVIDER == 'flux-schnell':
    from getpass import getpass
    HF_TOKEN = getpass('Hugging Face 읽기 권한 토큰 입력: ').strip()
    assert HF_TOKEN, 'FLUX 다운로드에는 Hugging Face 토큰이 필요합니다.'
    RUNTIME_ENV['HF_TOKEN'] = HF_TOKEN
    run_or_show([sys.executable, '-m', 'scripts.download_models', '--models', 'flux'], env=RUNTIME_ENV)
    assert Path('models/flux-schnell/model_index.json').is_file(), 'FLUX 모델 다운로드가 완료되지 않았습니다.'
    print('FLUX.1 Schnell 모델 확인 완료')
else:
    print('Sana 1.6B를 선택했으므로 FLUX 토큰과 다운로드를 건너뜁니다.')

## 음식 사진 업로드

음식과 용기가 보이는 JPG·PNG·WEBP 파일 한 장을 선택합니다.

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display
import shutil

uploaded = files.upload()
assert len(uploaded) == 1, '검증을 위해 음식 사진 한 장만 선택하세요.'
source_path = Path(next(iter(uploaded)))
suffix = source_path.suffix.lower()
assert suffix in {'.jpg', '.jpeg', '.png', '.webp'}, f'지원하지 않는 형식입니다: {suffix}'
input_path = Path('data/input') / f'example{suffix}'
input_path.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source_path), input_path)
display(Image.open(input_path))
print(f'입력 파일: {input_path}')

In [ ]:
import json
metadata = {
    'business_type': 'cafe',  # cafe | bakery | dessert | restaurant | pub
    'food_category': 'dessert',
    'foreground_position': 'center_lower',
    'light_direction': 'left',
}
metadata_path = Path('data/input/example_metadata.json')
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(metadata_path.read_text(encoding='utf-8'))

In [ ]:
# YOLO가 음식·용기를 찾지 못하는 경우에도 중앙 전경 상자로 전체 단계를 검증합니다.
run_or_show([sys.executable, '-m', 'scripts.run_background_replacement', '--input', str(input_path), '--metadata', str(metadata_path), '--enable-matting', '--enable-background-generator'], env=RUNTIME_ENV)

In [ ]:
from IPython.display import Image as DisplayImage
output_path = Path('data/output') / f'{input_path.stem}_background_replaced.jpg'
report_path = Path('data/reports') / f'{input_path.stem}_background_replacement_report.json'
assert output_path.is_file(), f'결과 파일이 없습니다: {output_path}'
assert report_path.is_file(), f'보고서 파일이 없습니다: {report_path}'
display(DisplayImage(filename=str(output_path)))
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report.get('stages', report), ensure_ascii=False, indent=2))
if 'step_2_detection_fallback' in report.get('stages', {}):
    print('주의: 탐지 대체 경로가 사용되었습니다. 운영용 결과에서는 음식·용기 특화 탐지 모델을 추가해야 합니다.')